**Use the visual spectrum images to initialize the sticher and tehn you can use the initialized sticher to 
create panoramas in different sensors like thermal and Night-vission**

In [ ]:

import cv2 
image_paths=[
r"D:\arma3_screenshots\videoFrames\Squared\camera (1).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (2).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (3).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (4).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (5).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (6).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (7).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (8).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (9).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (10).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (11).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (12).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (13).png",



]
# initialized a list of images 
imgs = [] 
  
for i in range(len(image_paths)): 
    imgs.append(cv2.imread(image_paths[i])) 
    imgs[i]=cv2.resize(imgs[i],(0,0),fx=0.5,fy=0.5) 
    # this is optional if your input images isn't too large 
    # you don't need to scale down the image 
    # in my case the input images are of dimensions 3000x1200 
    # and due to this the resultant image won't fit the screen 
    # scaling down the images  
# showing the original pictures 

  
stitchy=cv2.Stitcher.create() 
(dummy,output)=stitchy.stitch(imgs) 
  
if dummy != cv2.STITCHER_OK: 
  # checking if the stitching procedure is successful 
  # .stitch() function returns a true value if stitching is  
  # done successfully 
    print("stitching ain't successful") 
else:  
    print('Your Panorama is ready!!!') 
  
# final output 
#cv2.imwrite(r"C:\Users\user1\Desktop\panoVIS.png",output)
cv2.imshow('final result',cv2.resize(output,(0,0),fx=0.5,fy=0.5)) 
  
cv2.waitKey(0)
cv2.destroyAllWindows()


Your Panorama is ready!!!


In [31]:
import cv2
import numpy as np

# Load ArUco dictionary and detector parameters
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_50)
aruco_params = cv2.aruco.DetectorParameters()
# Create the detector
detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)
# Load images
image_paths = [r"D:\arma3_screenshots\videoFrames\Squared\camera (1).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (2).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (3).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (4).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (5).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (6).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (7).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (8).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (9).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (10).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (11).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (12).png",
r"D:\arma3_screenshots\videoFrames\Squared\camera (13).png",]  # your 12 images
images = [cv2.imread(path) for path in image_paths]

# Detect ArUco markers in each image
all_corners = []
all_ids = []

for img in images:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    corners, ids, _ = detector.detectMarkers(gray)
    all_corners.append(corners)
    all_ids.append(ids)

# Function to match ArUco corners between two images
def match_markers(corners1, ids1, corners2, ids2):
    if ids1 is None or ids2 is None:
        return None, None
    matched_pts1 = []
    matched_pts2 = []
    for i, id1 in enumerate(ids1.flatten()):
        if id1 in ids2:
            j = np.where(ids2.flatten() == id1)[0][0]
            # Use center of each marker as keypoint (you could use all 4 corners instead)
            c1 = corners1[i][0]
            c2 = corners2[j][0]
            center1 = np.mean(c1, axis=0)
            center2 = np.mean(c2, axis=0)
            matched_pts1.append(center1)
            matched_pts2.append(center2)
    if len(matched_pts1) >= 4:
        return np.array(matched_pts1), np.array(matched_pts2)
    else:
        return None, None

# Choose the middle image as the reference
center_idx = len(images) // 2
H_matrices = [None] * len(images)
H_matrices[center_idx] = np.eye(3)  # Identity for the center image

# Compute homographies to the center image
# Forward from center to right
for i in range(center_idx, len(images) - 1):
    pts1, pts2 = match_markers(all_corners[i], all_ids[i], all_corners[i + 1], all_ids[i + 1])
    if pts1 is not None:
        H, _ = cv2.findHomography(pts2, pts1, cv2.RANSAC)
        H_matrices[i + 1] = H_matrices[i] @ H

# Backward from center to left
for i in range(center_idx, 0, -1):
    pts1, pts2 = match_markers(all_corners[i], all_ids[i], all_corners[i - 1], all_ids[i - 1])
    if pts1 is not None:
        H, _ = cv2.findHomography(pts2, pts1, cv2.RANSAC)
        H_matrices[i - 1] = H_matrices[i] @ H

# Warp all images into panorama space
# First compute size of final canvas
sizes = []
for img, H in zip(images, H_matrices):
    if H is None:
        continue
    h, w = img.shape[:2]
    corners = np.array([[0,0],[w,0],[w,h],[0,h]], dtype=np.float32).reshape(-1,1,2)
    warped_corners = cv2.perspectiveTransform(corners, H)
    sizes.append(warped_corners)

all_corners = np.vstack(sizes)
[xmin, ymin] = np.int32(all_corners.min(axis=0).ravel() - 0.5)
[xmax, ymax] = np.int32(all_corners.max(axis=0).ravel() + 0.5)

# Compute translation to keep images in positive coordinates
translate = np.array([[1, 0, -xmin],
                      [0, 1, -ymin],
                      [0, 0,     1]])

# Create panorama canvas
panorama_size = (xmax - xmin, ymax - ymin)
panorama = np.zeros((panorama_size[1], panorama_size[0], 3), dtype=np.uint8)

# Warp and blend images
for img, H in zip(images, H_matrices):
    if H is None:
        continue
    warped = cv2.warpPerspective(img, translate @ H, panorama_size)
    mask = (warped > 0).astype(np.uint8) * 255
    panorama = cv2.seamlessClone(warped, panorama, mask[:,:,0], (panorama.shape[1]//2, panorama.shape[0]//2), cv2.NORMAL_CLONE)

#cv2.imwrite(r"C:\Users\user1\Desktop\panoVIS.png",panorama)
cv2.imshow('final result',cv2.resize(panorama,(0,0),fx=0.5,fy=0.5)) 
  
cv2.waitKey(0)
cv2.destroyAllWindows()


ValueError: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

In [29]:
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
aruco_params = cv2.aruco.DetectorParameters()

In [48]:

def match_markers(corners1, ids1, corners2, ids2):
    if ids1 is None or ids2 is None:
        return None, None

    matched_pts1 = []
    matched_pts2 = []

    ids1 = ids1.flatten()
    ids2 = ids2.flatten()

    for i, id1 in enumerate(ids1):
        if id1 in ids2:
            j = np.where(ids2 == id1)[0][0]
            # Use all 4 corners of each marker (in consistent order)
            pts1 = corners1[i][0]  # shape: (4, 2)
            pts2 = corners2[j][0]
            for k in range(4):  # 4 corners
                matched_pts1.append(pts1[k])
                matched_pts2.append(pts2[k])

    if len(matched_pts1) >= 4:
        return np.array(matched_pts1), np.array(matched_pts2)
    else:
        return None, None

In [57]:
i=8
match_markers(all_corners[i], all_ids[i], all_corners[i + 1], all_ids[i + 1])

IndexError: list index out of range